# PhantomVision — Local PC Quickstart

Run PhantomVision on **your own machine** (Windows, macOS, or Linux).

### Prerequisites
| Requirement | Minimum | Recommended |
|---|---|---|
| Python | 3.9+ | 3.10+ |
| PyTorch | 2.0+ | latest stable |
| RAM | 8 GB | 16 GB+ |
| GPU | CPU works | NVIDIA GPU with CUDA |

### Supported Datasets
| Source | Layout |
|--------|--------|
| Phantom Factory – Object Detection | `yolo/images/{train,val}.7z` + `labels/{train,val}.7z` |
| Phantom Factory – Keypoints | Same YOLO layout |
| Phantom Factory – Classification | `classifier/evolving_ds_*/{train,val}.7z` |
| COCO Export | `{train,val}.7z` with `coco_instances.json` |
| Plain YOLO / COCO | Already extracted on disk |
| No dataset | Runs with synthetic demo data |

Archives: `.zip` `.7z` `.tar.gz` `.tar.bz2` `.tar.xz` `.rar`

## 1 — Environment Setup

In [ ]:
import subprocess
import sys
from pathlib import Path

NOTEBOOK_DIR = Path(".").resolve()
REPO_ROOT = NOTEBOOK_DIR.parent if (NOTEBOOK_DIR.parent / "pyproject.toml").exists() else NOTEBOOK_DIR

def in_venv():
    return sys.prefix != sys.base_prefix

if not in_venv():
    venv = REPO_ROOT / ".venv"
    if not venv.exists():
        print("Creating virtual environment ...")
        subprocess.check_call([sys.executable, "-m", "venv", str(venv)])
        pip = str(venv / "bin" / "pip")
        print("Installing PhantomVision + extras ...")
        subprocess.check_call([pip, "install", "--upgrade", "pip", "setuptools", "wheel"], stdout=subprocess.DEVNULL)
        subprocess.check_call([pip, "install", "-e", str(REPO_ROOT)], stdout=subprocess.DEVNULL)
        subprocess.check_call([pip, "install", "tqdm", "py7zr", "rarfile", "jupyter", "ipykernel"], stdout=subprocess.DEVNULL)
        python = str(venv / "bin" / "python")
        subprocess.check_call([python, "-m", "ipykernel", "install", "--user", "--name", "phantomvision", "--display-name", "PhantomVision"], stdout=subprocess.DEVNULL)
    print("=" * 60)
    print("  Virtual environment ready at:", venv)
    print("  Switch the Jupyter kernel to 'PhantomVision' and re-run.")
    print("=" * 60)
else:
    for pkg in ["tqdm", "py7zr", "rarfile"]:
        try:
            __import__(pkg)
        except ImportError:
            subprocess.check_call([sys.executable, "-m", "pip", "install", pkg], stdout=subprocess.DEVNULL)
    print("Environment OK:", sys.prefix)

## 2 — Device Check

In [ ]:
import torch

print(f"PyTorch : {torch.__version__}")
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    gpu_mem = torch.cuda.get_device_properties(0).total_mem / 1024**3
    print(f"GPU     : {gpu_name}  ({gpu_mem:.1f} GB)")
    print(f"CUDA    : {torch.version.cuda}")
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    torch.backends.cudnn.benchmark = True
    DEVICE = torch.device("cuda:0")
elif hasattr(torch.backends, 'mps') and torch.backends.mps.is_available():
    print("Device  : Apple MPS")
    DEVICE = torch.device("mps")
else:
    print("Device  : CPU (training will be slow)")
    DEVICE = torch.device("cpu")

## 3 — Configure Training

Set `DATASET_PATH` to your dataset archive or folder.
Leave empty (`""`) to run a quick demo with synthetic data.

In [ ]:
# ╔════════════════════════════════════════════════════════════╗
# ║                    CONFIGURE HERE                         ║
# ╚════════════════════════════════════════════════════════════╝

DATASET_PATH = ""    # "/path/to/dataset" or "/path/to/data.7z"
TASK = "detection"    # "detection" | "keypoints" | "classification"
EPOCHS = 100
BATCH_SIZE = 0        # 0 = auto from GPU VRAM
IMG_SIZE = 640
LR = 0.01
MODEL = "resnext"     # "resnext" or "convnext"

## 4 — Prepare Dataset

Auto-detects format, extracts archives, converts YOLO to COCO if needed.

In [ ]:
import importlib.util
import json
import shutil
import textwrap

import numpy as np
from PIL import Image

spec = importlib.util.spec_from_file_location("linux_train", str(REPO_ROOT / "notebooks" / "linux_train.py"))
lt = importlib.util.module_from_spec(spec)
spec.loader.exec_module(lt)

WORKSPACE = Path("phantomvision_workspace").resolve()
WORKSPACE.mkdir(parents=True, exist_ok=True)

USE_REAL_DATA = bool(DATASET_PATH) and Path(DATASET_PATH).exists()

if USE_REAL_DATA:
    dataset_path = Path(DATASET_PATH).resolve()
    if dataset_path.is_file() and lt.is_archive(dataset_path):
        extract_dest = WORKSPACE / "dataset"
        if extract_dest.exists():
            shutil.rmtree(extract_dest)
        dataset_root = lt.extract_archive(dataset_path, extract_dest)
    else:
        dataset_root = dataset_path

    dataset_root = lt.resolve_dataset_root(dataset_root, TASK)
    fmt = lt.detect_format(dataset_root)
    print(f"Detected format: {fmt}")

    if fmt == "phantom_yolo":
        cls = dataset_root / "classes.txt"
        data_info = lt.prepare_phantom_yolo(dataset_root, WORKSPACE, cls if cls.exists() else None)
    elif fmt == "phantom_classifier":
        data_info = lt.prepare_phantom_classifier(dataset_root, WORKSPACE)
    elif fmt == "coco_archive":
        data_info = lt.prepare_coco_archive(dataset_root, WORKSPACE)
    elif fmt == "yolo_flat":
        data_info = lt.prepare_yolo_flat(dataset_root, WORKSPACE)
    elif fmt == "coco_flat":
        data_info = lt.prepare_coco_flat(dataset_root, WORKSPACE)
    else:
        raise RuntimeError(f"Unknown format in {dataset_root}")

    num_classes = lt.detect_num_classes(data_info["train_ann_file"])
    num_train = lt.count_images(data_info["train_ann_file"])
    print(f"Classes: {num_classes}  |  Train images: {num_train}")
else:
    print("No DATASET_PATH — creating synthetic demo data.")
    ROOT = Path("phantomvision_demo")
    CONFIGS = ROOT / "configs"
    DATA = ROOT / "data"
    IMGS = DATA / "images"
    CONFIGS.mkdir(parents=True, exist_ok=True)
    IMGS.mkdir(parents=True, exist_ok=True)

    (CONFIGS / "resnext_nano.yaml").write_text(textwrap.dedent("""model:\n  name: PhantomResNeXt-Nano\n  type: resnext\n  backbone: resnext_nano\n  neck_channels: 64\n  head_depth: 2\n  use_ghost: true\n  num_classes: 80\n  image_size: 640\ntraining:\n  epochs: 3\n  base_lr: 0.01\n  warmup_epochs: 1\n  accumulation_steps: 1\n  early_stopping_patience: 30\n"))
    (CONFIGS / "convnext_nano.yaml").write_text(textwrap.dedent("""model:\n  name: PhantomVision-Nano\n  backbone: convnext_tiny\n  channels: 64\n  transformer_depth: 2\n  token_count: 100\n  image_size: 640\ntraining:\n  epochs: 3\n"))
    coco = {"images": [], "annotations": [], "categories": [{"id": 1, "name": "object"}]}
    ann_id = 0
    for i in range(4):
        fname = f"img_{i:04d}.jpg"
        img = Image.fromarray(np.random.randint(0, 255, (640, 640, 3), dtype=np.uint8))
        img.save(str(IMGS / fname))
        coco["images"].append({"id": i, "file_name": fname, "width": 640, "height": 640})
        for _ in range(2):
            x, y = int(np.random.randint(0, 400)), int(np.random.randint(0, 400))
            w, h = int(np.random.randint(30, 200)), int(np.random.randint(30, 200))
            coco["annotations"].append({"id": ann_id, "image_id": i, "category_id": 1, "bbox": [x, y, w, h], "area": w * h, "iscrowd": 0})
            ann_id += 1
    (DATA / "annotations.json").write_text(json.dumps(coco))
    (CONFIGS / "data.yaml").write_text(f'img_dir: "{IMGS.resolve()}"\nann_file: "{(DATA / "annotations.json").resolve()}"\n')
    print(f"Synthetic data at: {ROOT.resolve()}")

## 5 — Train

Progress bar per batch + training snapshot per epoch.

In [ ]:
if USE_REAL_DATA:
    batch_size = BATCH_SIZE
    if batch_size <= 0 and torch.cuda.is_available():
        gpu_mem_gb = torch.cuda.get_device_properties(0).total_mem / 1024**3
        batch_size = lt.auto_batch_size(gpu_mem_gb, IMG_SIZE)
        print(f"Auto batch size: {batch_size}")
    elif batch_size <= 0:
        batch_size = 2

    model_cfg, data_cfg = lt.write_configs(
        WORKSPACE, data_info, num_classes,
        epochs=EPOCHS, batch_size=batch_size,
        img_size=IMG_SIZE, lr=LR, model_type=MODEL,
    )
    lt.run_training(model_cfg, data_cfg, data_info, DEVICE,
                    epochs=EPOCHS, batch_size=batch_size)
else:
    from phantomvision import PhantomVision
    model = PhantomVision(str(CONFIGS / "resnext_nano.yaml"))
    model.train(data=str(CONFIGS / "data.yaml"), task="detection")

## 6 — Inference & Benchmark

In [ ]:
import glob
import os

if not USE_REAL_DATA:
    # Model profiling
    from phantomvision.models.phantom_resnext import PhantomResNeXtModel
    from phantomvision.utils.profiler import model_summary
    for variant in ["resnext_pico", "resnext_nano", "resnext_small"]:
        m = PhantomResNeXtModel(variant=variant, num_classes=80)
        s = model_summary(m)
        print(f"{variant:>16s}  |  {s['params_M']}M  |  {s['flops_G']} GFLOPs  |  {s['size_mb']} MB")

    # Inference
    preds = model.predict(str(IMGS / "img_0000.jpg"))
    for key, val in preds.items():
        print(f"{key}: shape={val.shape}, dtype={val.dtype}")

    # Benchmark
    results = model.benchmark(warmup=3, runs=20)
    print(f"Latency: {results['avg_latency_ms']} ms  |  FPS: {results['fps']}")

    # Export
    model.export(format="onnx")
    size_mb = os.path.getsize("model.onnx") / (1024 * 1024)
    print(f"Exported model.onnx ({size_mb:.1f} MB)")

    # ConvNeXt
    from phantomvision import PhantomVision
    convnext = PhantomVision(str(CONFIGS / "convnext_nano.yaml"))
    preds = convnext.predict(str(IMGS / "img_0000.jpg"))
    for key, val in preds.items():
        print(f"{key}: shape={val.shape}")
else:
    runs = sorted(glob.glob("runs/train_*"))
    if runs:
        best = f"{runs[-1]}/checkpoint_best.pt"
        last = f"{runs[-1]}/checkpoint_last.pt"
        ckpt_path = best if Path(best).exists() else last
        if Path(ckpt_path).exists():
            ckpt = torch.load(ckpt_path, map_location="cpu")
            print(f"Best checkpoint: {ckpt_path}")
            print(f"Epoch: {ckpt.get('epoch', '?')}  |  Loss: {ckpt.get('best_loss', '?'):.4f}")

## 7 — CLI Alternative

You can also train from the terminal:

```bash
# With real dataset:
python notebooks/linux_train.py /path/to/dataset --task detection --epochs 100

# Or the local quickstart script:
python notebooks/local_quickstart.py /path/to/dataset --task detection
```

---

Set `DATASET_PATH` above and re-run to train on your data.

See the [README](https://github.com/Dillun-Holmes/AI_vision_model) for all model variants and options.